# Mini Search Engine Berita IHSG (CNBC Indonesia)
Final Project Information Retrieval

## Langkah 1: Instalasi
Tujuan: Menginstal dependensi pustaka yang dibutuhkan (`requirements.txt`).

In [ ]:
%pip install -r requirements.txt

## Langkah 2: Konfigurasi dan Impor
Tujuan: Mengimpor pustaka dan mendefinisikan konstanta/parameter terpusat (seed, path, batasan crawling, model dense, bobot RRF).

In [12]:
import hashlib
import json
import math
import os
import pickle
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

# Menetapkan seed acak demi reprodusibilitas eksperimen
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Konfigurasi crawler
CONTACT_EMAIL = "dickyferdy5@gmail.com"
TARGET_DOCS = 2000
CRAWL_DELAY = 2.0
MAX_FETCHES = 8000

# Konfigurasi retriever leksikal (BM25 Okapi)
BM25_K1 = 1.5
BM25_B = 0.75

# Konfigurasi retriever dense (multilingual-e5-small)
EMBED_MODEL = "intfloat/multilingual-e5-small"
EMBED_MAX_LEN = 256
EMBED_BATCH = 64

# Konfigurasi hybrid fusion & kedalaman evaluasi
RRF_K = 60
CANDIDATES = 100
POOL_DEPTH = 10
EVAL_K = 10

# Jalur direktori dan berkas (relatif terhadap direktori kerja notebook)
PATH_DATA_DIR = Path("data")
PATH_INDEXES_DIR = Path("indexes")
PATH_RESULTS_DIR = Path("results")

PATH_QUERIES = PATH_DATA_DIR / "queries.csv"
PATH_CORPUS = PATH_DATA_DIR / "corpus.jsonl"
PATH_CORPUS_URLS = PATH_DATA_DIR / "corpus_urls.txt"
PATH_POOL_TO_LABEL = PATH_DATA_DIR / "pool_to_label.csv"
PATH_QRELS = PATH_DATA_DIR / "qrels.csv"
PATH_VISITED_URLS = PATH_DATA_DIR / "visited_urls.txt"
PATH_KNOWN_URLS = PATH_DATA_DIR / "known_urls.txt"

PATH_BM25_TOKENS = PATH_INDEXES_DIR / "bm25_tokens.pkl"
PATH_FAISS_INDEX = PATH_INDEXES_DIR / "faiss.index"
PATH_DENSE_IDS = PATH_INDEXES_DIR / "dense_ids.json"

PATH_PER_QUERY = PATH_RESULTS_DIR / "per_query.csv"
PATH_SUMMARY = PATH_RESULTS_DIR / "summary.csv"
PATH_SIGNIFICANCE = PATH_RESULTS_DIR / "significance.csv"
PATH_ERROR_ANALYSIS = PATH_RESULTS_DIR / "error_analysis.csv"

# Memastikan seluruh direktori output tersedia
for path_dir in [PATH_DATA_DIR, PATH_INDEXES_DIR, PATH_RESULTS_DIR]:
    path_dir.mkdir(parents=True, exist_ok=True)

print("Konfigurasi dan pustaka berhasil dimuat. Direktori siap.")

Konfigurasi dan pustaka berhasil dimuat. Direktori siap.


## Langkah 3: Ambil Korpus (Crawler)
Tujuan: Mengumpulkan 2.000 artikel berita pasar modal/IHSG dari CNBC Indonesia secara terfokus, etis, dan idempoten.

In [13]:
import hashlib
import json
import re
import time
import urllib.parse
import xml.etree.ElementTree as ET
from pathlib import Path
import requests
from bs4 import BeautifulSoup

ARTICLE_URL_PATTERN = re.compile(r"^https://www\.cnbcindonesia\.com/market/\d{14}-\d+-\d+/[^/?#]+$")
DISALLOWED_SUBSTRINGS = ["?", "/embed", "/logout", "/search", "/api", "/partner", "/indeks/"]

def is_allowed_url(url: str) -> bool:
    """Memeriksa apakah URL mematuhi aturan konservatif robots.txt CNBC."""
    parsed = urllib.parse.urlparse(url)
    full_path_query = parsed.path + ("?" + parsed.query if parsed.query else "")
    for disallowed in DISALLOWED_SUBSTRINGS:
        if disallowed in full_path_query:
            return False
    return True

def extract_article(html: str, url: str) -> dict | None:
    """Mengekstrak judul, tanggal publikasi, dan isi berita dari HTML artikel CNBC Indonesia."""
    soup = BeautifulSoup(html, "html.parser")
    
    # 1. Judul dari <meta property="og:title"> dengan fallback ke <title>
    og_title = soup.find("meta", property="og:title")
    title = og_title.get("content", "").strip() if og_title else ""
    if not title:
        title_tag = soup.find("title")
        title = title_tag.get_text(strip=True) if title_tag else ""
        
    # 2. Tanggal publikasi dari JSON-LD datePublished
    date_published = ""
    for script in soup.find_all("script", type="application/ld+json"):
        if not script.string:
            continue
        try:
            data = json.loads(script.string.strip())
            items = data if isinstance(data, list) else [data]
            for item in items:
                if isinstance(item, dict):
                    if "datePublished" in item:
                        date_published = item["datePublished"]
                        break
                    for sub in item.get("@graph", []):
                        if isinstance(sub, dict) and "datePublished" in sub:
                            date_published = sub["datePublished"]
                            break
            if date_published:
                break
        except (json.JSONDecodeError, AttributeError):
            continue
            
    # 3. Ekstraksi teks isi dari div.detail-text terdalam
    detail_divs = soup.find_all("div", class_="detail-text")
    if not detail_divs:
        return None
    inner_detail = detail_divs[-1]
    
    paragraphs = []
    for p in inner_detail.find_all("p"):
        p_text = p.get_text(strip=True)
        if len(p_text) <= 30:
            continue
        if p_text.startswith("Baca:") or p_text.startswith("Foto:") or "Gambas:" in p_text:
            continue
        paragraphs.append(p_text)
        
    full_text = " ".join(paragraphs)
    
    # Menghapus awalan dateline 'Jakarta, CNBC Indonesia -'
    full_text = re.sub(r"^(?:Jakarta,\s*)?CNBC\s+Indonesia\s*[-–—]\s*", "", full_text, flags=re.IGNORECASE)
    full_text = re.sub(r"\s+", " ", full_text).strip()
    
    # Filter dokumen: > 300 karakter dan memuat 'ihsg' atau 'indeks harga saham gabungan'
    text_lower = full_text.lower()
    if len(full_text) <= 300:
        return None
    if "ihsg" not in text_lower and "indeks harga saham gabungan" not in text_lower:
        return None
        
    doc_id = "d" + hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
    return {
        "id": doc_id,
        "title": title,
        "date": date_published,
        "text": full_text,
        "url": url
    }

def run_crawler(target_docs=TARGET_DOCS, crawl_delay=CRAWL_DELAY, session=None):
    """Menjalankan crawling berita CNBC secara terfokus, etis, dan idempoten."""
    if not CONTACT_EMAIL or not CONTACT_EMAIL.strip():
        raise ValueError("Crawling ditolak: CONTACT_EMAIL wajib diisi di sel konfigurasi sebelum menjalankan crawler.")
        
    existing_docs = {}
    if PATH_CORPUS.exists():
        with open(PATH_CORPUS, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    doc = json.loads(line)
                    existing_docs[doc["url"]] = doc
        print(f"Korpus saat ini memuat {len(existing_docs)} artikel.")
        if len(existing_docs) >= target_docs:
            print(f"Korpus sudah mencapai target ({target_docs} artikel). Pengambilan data dilewati.")
            return
            
    if session is None:
        session = requests.Session()
        session.headers.update({
            "User-Agent": f"IR-Course-Bot/1.0 (tugas kuliah Information Retrieval; kontak: {CONTACT_EMAIL})"
        })
        
    visited_urls = set(existing_docs.keys())
    if PATH_VISITED_URLS.exists():
        with open(PATH_VISITED_URLS, "r", encoding="utf-8") as f:
            visited_urls.update(line.strip() for line in f if line.strip())
            
    known_urls = []
    if PATH_KNOWN_URLS.exists():
        with open(PATH_KNOWN_URLS, "r", encoding="utf-8") as f:
            known_urls = [line.strip() for line in f if line.strip() and line.strip() not in visited_urls]
            
    reproduce_mode = False
    if PATH_CORPUS_URLS.exists():
        with open(PATH_CORPUS_URLS, "r", encoding="utf-8") as f:
            target_urls = [line.strip() for line in f if line.strip()]
        needed_urls = [u for u in target_urls if u not in existing_docs]
        if needed_urls:
            print(f"Mode reproduksi aktif: mengambil {len(needed_urls)} artikel dari corpus_urls.txt...")
            known_urls = needed_urls
            reproduce_mode = True
            
    if not known_urls and not reproduce_mode:
        print("Memulai penemuan URL awal dari sitemap...")
        sitemaps = [
            "https://www.cnbcindonesia.com/market/sitemap_news.xml",
            "https://www.cnbcindonesia.com/market/sitemap_web.xml"
        ]
        for sm_url in sitemaps:
            try:
                resp = session.get(sm_url, timeout=10)
                if resp.status_code == 200:
                    root = ET.fromstring(resp.content)
                    ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}
                    for loc in root.findall(".//sm:loc", ns):
                        u = loc.text.strip() if loc.text else ""
                        if u and ARTICLE_URL_PATTERN.match(u) and is_allowed_url(u) and u not in visited_urls:
                            known_urls.append(u)
                time.sleep(crawl_delay)
            except Exception as e:
                print(f"Peringatan: Gagal membaca sitemap {sm_url}: {e}")
                
    known_urls.sort(key=lambda u: (0 if "ihsg" in u.lower() else 1))
    consecutive_errors = 0
    fetch_count = 0
    saved_count = len(existing_docs)
    
    print(f"Memulai loop crawler (target: {target_docs} artikel, max fetches: {MAX_FETCHES})...")
    
    with open(PATH_CORPUS, "a", encoding="utf-8") as corpus_file:
        while known_urls and saved_count < target_docs and fetch_count < MAX_FETCHES:
            url = known_urls.pop(0)
            if url in visited_urls:
                continue
            visited_urls.add(url)
            fetch_count += 1
            
            try:
                resp = session.get(url, timeout=10)
                if resp.status_code == 429:
                    print("HTTP 429: Rate limit tercapai, tidur 60 detik...")
                    time.sleep(60)
                    consecutive_errors += 1
                    continue
                if resp.status_code != 200:
                    consecutive_errors += 1
                    if consecutive_errors >= 8:
                        print("Mencapai 8 error beruntun. Menghentikan crawling.")
                        break
                    time.sleep(crawl_delay)
                    continue
                    
                consecutive_errors = 0
                html = resp.text
                
                if not reproduce_mode:
                    soup = BeautifulSoup(html, "html.parser")
                    for a_tag in soup.find_all("a", href=True):
                        href = a_tag["href"]
                        full_u = urllib.parse.urljoin(url, href).split("#")[0]
                        if ARTICLE_URL_PATTERN.match(full_u) and is_allowed_url(full_u):
                            if full_u not in visited_urls and full_u not in known_urls:
                                if "ihsg" in full_u.lower():
                                    known_urls.insert(0, full_u)
                                else:
                                    known_urls.append(full_u)
                                    
                doc = extract_article(html, url)
                if doc:
                    corpus_file.write(json.dumps(doc, ensure_ascii=False) + "\n")
                    corpus_file.flush()
                    saved_count += 1
                    if saved_count % 50 == 0 or saved_count == target_docs:
                        print(f"Tersimpan: {saved_count}/{target_docs} artikel (fetches: {fetch_count}).")
                        
                time.sleep(crawl_delay)
            except Exception as e:
                consecutive_errors += 1
                if consecutive_errors >= 8:
                    print(f"Mencapai 8 error beruntun ({e}). Menghentikan crawling.")
                    break
                time.sleep(crawl_delay)
                
    with open(PATH_VISITED_URLS, "w", encoding="utf-8") as f:
        for u in visited_urls:
            f.write(u + "\n")
    with open(PATH_KNOWN_URLS, "w", encoding="utf-8") as f:
        for u in known_urls:
            f.write(u + "\n")
            
    all_corpus_urls = []
    if PATH_CORPUS.exists():
        with open(PATH_CORPUS, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    all_corpus_urls.append(json.loads(line)["url"])
    with open(PATH_CORPUS_URLS, "w", encoding="utf-8") as f:
        for u in all_corpus_urls:
            f.write(u + "\n")
            
    print(f"Proses crawler selesai. Total artikel di korpus: {len(all_corpus_urls)}.")

run_crawler()


Memulai penemuan URL awal dari sitemap...
Memulai loop crawler (target: 2000 artikel, max fetches: 8000)...
Tersimpan: 50/2000 artikel (fetches: 50).
Tersimpan: 100/2000 artikel (fetches: 101).
Tersimpan: 150/2000 artikel (fetches: 175).
Tersimpan: 200/2000 artikel (fetches: 254).
Tersimpan: 250/2000 artikel (fetches: 332).
Tersimpan: 300/2000 artikel (fetches: 423).
Tersimpan: 350/2000 artikel (fetches: 489).
Tersimpan: 400/2000 artikel (fetches: 545).
Tersimpan: 450/2000 artikel (fetches: 606).
Tersimpan: 500/2000 artikel (fetches: 668).
Tersimpan: 550/2000 artikel (fetches: 739).
Tersimpan: 600/2000 artikel (fetches: 806).
Tersimpan: 650/2000 artikel (fetches: 895).
Tersimpan: 700/2000 artikel (fetches: 962).
Tersimpan: 750/2000 artikel (fetches: 1035).
Tersimpan: 800/2000 artikel (fetches: 1112).
Tersimpan: 850/2000 artikel (fetches: 1168).
Tersimpan: 900/2000 artikel (fetches: 1218).
Tersimpan: 950/2000 artikel (fetches: 1268).
Tersimpan: 1000/2000 artikel (fetches: 1318).
Tersimp

## Langkah 4: Praproses, Retriever, Bangun Indeks
Tujuan: Melakukan pembersihan teks (case folding, tokenisasi, normalisasi, stopword removal, stemming terproteksi) serta membangun indeks BM25 dan indeks dense FAISS.

In [ ]:
import re
from functools import lru_cache
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Stopword standar Sastrawi ditambah istilah web umum
_stopword_factory = StopWordRemoverFactory()
STOPWORDS = set(_stopword_factory.get_stop_words()) | {"cnbc", "com", "www", "http", "https"}

# Kamus normalisasi variasi istilah pasar modal dan mata uang
NORMALIZATION_MAP = {
    "idx": "bei",
    "dollar": "dolar",
    "usd": "dolar",
    "rp": "rupiah",
}

# Istilah domain yang dilindungi dari pemotongan imbuhan (stemming)
PROTECTED_TERMS = {
    "ihsg", "bei", "ojk", "bi", "msci", "ipo", "lq45", "emiten", "asing", "ssi"
}

_stemmer_factory = StemmerFactory()
_stemmer = _stemmer_factory.create_stemmer()

@lru_cache(maxsize=50000)
def cached_stem(word: str) -> str:
    """Melakukan stemming kata bahasa Indonesia dengan cache LRU untuk kecepatan."""
    return _stemmer.stem(word)

def preprocess(text: str) -> list[str]:
    """Melakukan praproses teks sesuai ketentuan SPEC 4.2:
    
    1. Huruf kecil (case folding)
    2. Menghapus tautan URL
    3. Tokenisasi [a-z][a-z0-9]* (hanya token diawali huruf a-z)
    4. Normalisasi istilah (idx->bei, dollar/usd->dolar, rp->rupiah)
    5. Membuang token 1 huruf dan stopword
    6. Stemming Sastrawi dengan lru_cache (kecuali istilah domain terproteksi)
    """
    if not text:
        return []

    text_lower = text.lower()
    text_no_url = re.sub(r"https?://\S+|www\.\S+", " ", text_lower)
    raw_tokens = re.findall(r"[a-z][a-z0-9]*", text_no_url)

    processed_tokens = []
    for token in raw_tokens:
        norm_token = NORMALIZATION_MAP.get(token, token)
        if len(norm_token) <= 1 or norm_token in STOPWORDS:
            continue

        if norm_token in PROTECTED_TERMS:
            stemmed = norm_token
        else:
            stemmed = cached_stem(norm_token)

        if len(stemmed) > 1 and stemmed not in STOPWORDS:
            processed_tokens.append(stemmed)

    return processed_tokens


In [ ]:
import pickle
from rank_bm25 import BM25Okapi

def load_corpus(corpus_path=PATH_CORPUS) -> list[dict]:
    """Memuat dokumen korpus dari berkas corpus.jsonl.
    
    Mengembalikan daftar objek dokumen [id, title, date, text, url].
    """
    if not corpus_path.exists():
        print(f"Peringatan: Berkas korpus {corpus_path} belum ditemukan.")
        return []
    docs = []
    with open(corpus_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                docs.append(json.loads(line))
    print(f"Berhasil memuat {len(docs)} dokumen dari {corpus_path}.")
    return docs

class BM25Retriever:
    """Retriever leksikal berbasis BM25 Okapi dengan antarmuka pencarian terstandarisasi."""
    def __init__(self, doc_ids: list[str], corpus_tokens: list[list[str]], k1: float = BM25_K1, b: float = BM25_B):
        self.doc_ids = doc_ids
        self.corpus_tokens = corpus_tokens
        self.k1 = k1
        self.b = b
        self.bm25 = BM25Okapi(corpus_tokens, k1=k1, b=b) if corpus_tokens else None

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        """Mencari k dokumen teratas berdasarkan kueri menggunakan skor BM25 Okapi.
        
        Mengembalikan list pasangan (docid, skor).
        Jika kueri tidak menghasilkan token setelah praproses, mengembalikan [].
        Hanya dokumen dengan skor lebih dari 0 yang dikembalikan.
        """
        if not self.bm25 or not self.doc_ids:
            return []

        query_tokens = preprocess(query)
        if not query_tokens:
            return []

        scores = self.bm25.get_scores(query_tokens)
        
        # Ambil hanya dokumen dengan skor positif
        positive_indices = [i for i, score in enumerate(scores) if score > 0]
        if not positive_indices:
            return []

        # Urutkan berdasarkan skor tertinggi (descending)
        positive_indices.sort(key=lambda i: scores[i], reverse=True)
        top_indices = positive_indices[:k]

        return [(self.doc_ids[i], float(scores[i])) for i in top_indices]

def build_or_load_bm25_index(corpus: list[dict], index_path=PATH_BM25_TOKENS) -> BM25Retriever:
    """Membangun indeks BM25 jika belum ada atau memuat dari berkas token pickle secara idempoten."""
    if not corpus:
        print("Korpus kosong. Indeks BM25 tidak dibangun.")
        return BM25Retriever([], [])

    corpus_ids = [doc["id"] for doc in corpus]

    # Cek apakah berkas indeks sudah ada dan sesuai dengan korpus saat ini
    if index_path.exists():
        try:
            with open(index_path, "rb") as f:
                saved_data = pickle.load(f)
            if saved_data.get("ids") == corpus_ids:
                print(f"Memuat indeks BM25 yang sudah ada dari {index_path} ({len(corpus_ids)} dokumen).")
                return BM25Retriever(saved_data["ids"], saved_data["tokens"])
            else:
                print("Daftar dokumen korpus berubah. Membangun ulang indeks BM25...")
        except Exception as e:
            print(f"Gagal memuat {index_path} ({e}). Membangun ulang indeks...")

    print(f"Melakukan praproses dan pembuatan indeks BM25 untuk {len(corpus)} dokumen...")
    # Teks yang diindeks: judul + ". " + isi
    corpus_tokens = [preprocess(doc["title"] + ". " + doc["text"]) for doc in corpus]

    # Simpan {"ids", "tokens"} dengan pickle
    with open(index_path, "wb") as f:
        pickle.dump({"ids": corpus_ids, "tokens": corpus_tokens}, f)

    print(f"Indeks BM25 berhasil disimpan ke {index_path}.")
    return BM25Retriever(corpus_ids, corpus_tokens)

# Memuat korpus dan membangun indeks BM25
corpus = load_corpus()
bm25_retriever = build_or_load_bm25_index(corpus)


## Langkah 5: Muat Sistem, Ukur Memori, Fungsi Cari
Tujuan: Memuat model dan indeks, mengestimasi footprint RAM/RSS proses, serta menyiapkan fungsi antarmuka seragam `search(query, k)` untuk BM25, Dense, dan Hybrid RRF.

## Langkah 6: Pooling
Tujuan: Mengambil top-10 kandidat dokumen dari BM25, Dense, dan Hybrid untuk setiap kueri uji, menggabungkan secara unik, dan mengacaknya ke `data/pool_to_label.csv`.

## Langkah 7: Pelabelan Manual dan Pembaca Qrels
Tujuan: Membaca hasil relevance judgement manual manusia secara andal (`read_csv_robust`) dan menghasilkan ground truth `data/qrels.csv`.

## Langkah 8: Evaluasi Efektivitas
Tujuan: Menghitung metrik performa retrieval (nDCG@10, MAP, MRR, P@10) per kueri dan rata-rata sistem untuk membandingkan BM25 vs Dense vs Hybrid RRF.

## Langkah 9: Evaluasi Efisiensi
Tujuan: Mengukur efisiensi sistem berupa latensi pencarian (dengan fase pemanasan dan rotasi bergilir), konsumsi memori RSS, serta ukuran berkas indeks di disk.

## Langkah 10: Signifikansi dan Analisis Error
Tujuan: Melakukan uji signifikansi statistik (paired t-test dan Wilcoxon) antarmetode serta memetakan kueri terbaik dan terburuk pada sistem hybrid.

## Langkah 11: Antarmuka
Tujuan: Menyediakan widget pencarian interaktif di dalam notebook (`ipywidgets`) dengan opsi pemilihan metode dan visualisasi cuplikan hasil.